In [ ]:
import matplotlib
from matplotlib import pyplot as plt
from pylab import *

import numpy as np
import os.path as osp
import os,sys

from tqdm.notebook import tqdm
import time

Install SISL and ASE

In [ ]:
!pip install -q sisl[viz] ase

In [ ]:
import sisl as si
from ase.visualize import view

In [ ]:
from scipy.spatial import cKDTree
from scipy.optimize import differential_evolution
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

In [ ]:
si.__version__

In [ ]:
# Physical parameters

# Carbon Carbon bond distance
a_cc = 1.44
# Carbon Carbon hopping
t=2.7
# Carbon onsite
e0=0

# Transport with `sisl`: a beginner's tutorial

This tutorial introduces the basic ingredients of quantum transport calculations with **sisl**.
The tutorial is divided into three parts:

1. **Nanoporous graphene (NPG): from DFT to a reduced $p_z$ Hamiltonian.** We read a precomputed SIESTA Hamiltonian, inspect its band structure, keep only the carbon and nitrogen $p_z$ orbitals, compare the reduced model with the original DFT bands, visualize selected wave functions, and calculate the transmission along the conducting direction.
2. **Real-space self-energies (RSSE) for graphene.** We build a simple graphene tight-binding model and use a real-space self-energy to embed a finite graphene region in an otherwise infinite graphene sheet. We then calculate the DOS and LDOS from the retarded Green function.
3. **Vertical graphene–CNT–graphene transport.** We construct a carbon nanotube (CNT) connected between two graphene sheets, reuse the graphene RSSEs as electrodes, and calculate transmission, DOS, and spatially resolved LDOS.


Get some data

In [ ]:
!test -d /content/NM4QM-2026-demo || git clone --depth 1 https://github.com/mbrandbyge-DTU/NM4QM-2026-demo.git /content/NM4QM-2026-demo

%cd /content/NM4QM-2026-demo

# I. Nanoporous graphene: reducing a DZP Hamiltonian to a $p_z$ model

## 1. Read the SIESTA Hamiltonian

We begin with a **precomputed DFT Hamiltonian from SIESTA**. `sisl` can read the Hamiltonian directly from the SIESTA input/output files. The resulting object contains both the atomic geometry and the Hamiltonian/overlap matrices needed for the following analysis.

In [ ]:
dir = './NPG_DZP'
# Read Hamiltonian
H_dft = si.get_sile(dir+'/RUN.fdf').read_hamiltonian()

Before doing any electronic-structure or transport calculation, it is useful to inspect the geometry. Here we use `sisl`'s plotting tools to visualize the nanoporous graphene (NPG) structure.

The NPG consists of laterally connected graphene nanoribbons (GNRs). In this example, every second GNR is chemically modified with nitrogen atoms at the bridges.

In [ ]:
H_dft.geometry.plot(axes='xy',backend='matplotlib')

The same geometry can also be converted to an ASE `Atoms` object and opened with an ASE-compatible viewer. This is useful for interactive inspection of larger structures.

In [ ]:
view(H_dft.to.ase())

Next we calculate the **band structure** of the original DFT Hamiltonian. A band-structure calculation requires a path through reciprocal space and a number of $k$ points along that path. Here the path is $X \rightarrow\Gamma
\rightarrow Y$.

In [ ]:
nk=30
bs = si.BandStructure(H_dft, [[0.5, 0, 0], [0, 0, 0], [0, 0.5, 0]], nk, [r'$X$', r'$\Gamma$', r'$Y$'])
lk, kt, kl = bs.lineark(True)

In [ ]:
# To get the bands eigenvalues
bs_dft =bs.apply.array.eigh()

In [ ]:
fig, ax = plt.subplots(figsize=(4, 6))

firsttime= True
for bk in bs_dft.T:
    if(firsttime):
        ax.plot(lk, bk, 'k-', label='DZP')
        firsttime=False
    else:
        ax.plot(lk, bk, 'k')

xcoords = kt
for k in xcoords:
    plt.axvline(x=k,c='k')
plt.xticks(kt, kl)
ax.legend(loc='upper right',fontsize=12,framealpha=1)
plt.ylim(-1.5,1.5)
plt.xlim(kt[0],kt[2])
plt.ylabel(r'$E-E_F$ (eV)',fontsize=16)

## 2. Keep only the $p_z$ orbitals

The full DZP basis contains several atomic orbitals per atom. For the low-energy $\pi$ bands of graphene-like systems, the dominant contribution usually comes from the out-of-plane $p_z$ orbitals.

We therefore construct a reduced, tight-binding-like Hamiltonian by keeping only the carbon and nitrogen $p_z$ orbitals.

In [ ]:
# Atoms whose orbitals will be extracted from DFT
C_list = (H_dft.atoms.Z == 6).nonzero()[0]
C_atom = H_dft.geometry.atoms[C_list[0]]
Corbs = C_atom.orbitals

N_list = (H_dft.atoms.Z == 7).nonzero()[0]
N_atom = H_dft.geometry.atoms[N_list[0]]
Norbs  = N_atom.orbitals

print('Nitrogen')
print('orbital number orbtial name')
for i,orb in enumerate(N_atom.orbitals):
    print(i,orb.name())
print('##################')
print('Carbon')
print('orbital number orbtial name')
for i,orb in enumerate(C_atom.orbitals):
    print(i,orb.name())

`sub` can be used to keep a selected subset of atoms, while `sub_orbital` keeps selected orbitals on those atoms.

First we retain only carbon and nitrogen atoms. We then retain orbital index `3`, which was identified above as the relevant $p_z$ orbital for this basis. The result is a much smaller Hamiltonian that is easier to analyze and use in transport calculations.

In [ ]:
# List of Carbon and Nitrogen
CN_list = np.array(list(C_list)+list(N_list))
H_tb = H_dft.sub(CN_list)
H_tb.reduce()
H_tb = H_tb.sub_orbital(range(H_tb.geometry.na),[3])
H_tb.reduce()

In [ ]:
print("Number of atoms:",H_dft.na)
print("Number of orbitals:",H_dft.no)

In [ ]:
print("Number of atoms:",H_tb.na)
print("Number of orbitals:",H_tb.no)

After pruning the basis, the absolute energy reference of the reduced Hamiltonian does not necessarily coincide with that of the original DFT Hamiltonian. We therefore apply a rigid energy shift so that the valence-band edge of the reduced model aligns with the corresponding DFT state.

In [ ]:
tmp = H_dft.eigh()
en_dft = tmp[tmp<0].max()
tmp = H_tb.eigh()
en_tb = tmp[tmp<0].max()
print(en_dft, en_tb)

eshift = en_dft - en_tb
H_tb.shift(eshift)
print("Energy shift: ",eshift)

In [ ]:
nk=30
band_tb = si.BandStructure(H_tb, [[0.5, 0, 0], [0, 0, 0], [0, 0.5, 0]],nk, [r'$X$', r'$\Gamma$', r'$Y$'])
lk, kt, kl = band_tb.lineark(True)
bs_tb =band_tb.apply.array.eigh()

In [ ]:
fig, ax = plt.subplots(figsize=(4, 6))

firsttime=True
for bk in bs_tb.T:
    bk=bk
    if(firsttime):
        ax.plot(lk, bk, 'r--', label=r'$p_z$TB')
        firsttime=False
    else:
        ax.plot(lk, bk, 'r--')

firsttime= True
for bk in bs_dft.T:
    if(firsttime):
        ax.plot(lk, bk, 'k-', label='full')
        firsttime=False
    else:
        ax.plot(lk, bk, 'k')

xcoords = kt
for k in xcoords:
    plt.axvline(x=k,c='k')
plt.xticks(kt, kl)
ax.legend(loc='upper right',fontsize=12,framealpha=1)
plt.ylim(-1.5,1.5)
plt.xlim(kt[0],kt[2])
plt.ylabel(r'$E-E_F$ (eV)',fontsize=16)

### Exercise

The $p_z$-only model is intentionally minimal. Compare it with the full DFT bands and identify the energy range where it works well. Which additional orbitals would you keep if you wanted a better description farther away from the Fermi level?

In [ ]:
# List of Carbon and Nitrogen
CN_list = np.array(list(C_list)+list(N_list))
H_tb = H_dft.sub(CN_list)
H_tb.reduce()
H_tb = H_tb.sub_orbital(range(H_tb.geometry.na),[3,
                                                 #fill here,
                                                6,9,11

                                                ])
H_tb.reduce()

In [ ]:
tmp = H_dft.eigh()
en_dft = tmp[tmp<0].max()
tmp = H_tb.eigh()
en_tb = tmp[tmp<0].max()
print(en_dft, en_tb)

eshift = en_dft - en_tb
H_tb.shift(eshift)
print("Energy shift: ",eshift)

In [ ]:
nk=30
band_tb = si.BandStructure(H_tb, [[0.5, 0, 0], [0, 0, 0], [0, 0.5, 0]],nk, [r'$X$', r'$\Gamma$', r'$Y$'])
lk, kt, kl = band_tb.lineark(True)
bs_tb =band_tb.apply.array.eigh()

In [ ]:
fig = plt.figure('Band comparison',figsize=(4,6))
ax = fig.add_subplot()

firsttime=True
for bk in bs_tb.T:
    bk=bk
    if(firsttime):
        ax.plot(lk, bk, 'r--', label=r'$p_z$TB')
        firsttime=False
    else:
        ax.plot(lk, bk, 'r--')

firsttime= True
for bk in bs_dft.T:
    if(firsttime):
        ax.plot(lk, bk, 'k-', label='full')
        firsttime=False
    else:
        ax.plot(lk, bk, 'k')

xcoords = kt
for k in xcoords:
    plt.axvline(x=k,c='k',lw=0.1)

plt.xticks(kt, kl)
ax.legend(loc='upper right',fontsize=12,framealpha=1)
plt.ylim(-1.5,1.5)
plt.xlim(kt[0],kt[2])
plt.ylabel(r'$E-E_F$ (eV)',fontsize=16)

## 3. Visualize selected electronic states

We now use the reduced $p_z$ Hamiltonian to visualize states close to the valence- and conduction-band edges. `sisl` evaluates the eigenstate wave function on a real-space grid, which allows us to see where the selected state is localized within the NPG unit cell.

In [ ]:
# List of Carbon and Nitrogen
CN_list = np.array(list(C_list)+list(N_list))
H_tb = H_dft.sub(CN_list)
H_tb.reduce()
H_tb = H_tb.sub_orbital(range(H_tb.geometry.na),[3])
H_tb.reduce()

In [ ]:
tmp = H_dft.eigh()
en_dft = tmp[tmp<0].max()
tmp = H_tb.eigh()
en_tb = tmp[tmp<0].max()
print(en_dft, en_tb)

eshift = en_dft - en_tb
H_tb.shift(eshift)
print("Energy shift: ",eshift)

In [ ]:
es = H_tb.eigenstate(dtype=np.float64).sub([len(H_tb) // 2 ])
grid_spacing = 0.2
g = si.Grid(grid_spacing, dtype=np.float64, geometry=H_tb.geometry)
es.wavefunction(g)

In [ ]:
z_idx = g.shape[2] // 2
x, y = np.mgrid[:g.shape[0], :g.shape[1]]
x = x * g.dcell[0, 0] + y * g.dcell[1, 0]
y = y * g.dcell[0, 1] + y * g.dcell[1, 1]

fig, ax = plt.subplots(figsize=(8, 6))
ax.set_aspect('equal')
contour = ax.contourf(x, y,np.abs(g.grid[:, :, z_idx]), cmap="inferno", levels=50)

xyz = H_tb.geometry.xyz
ax.scatter(H_tb.xyz[H_tb.atoms.Z==6, 0], xyz[H_tb.atoms.Z==6, 1], 40, c='gray', label="C")
ax.scatter(H_tb.xyz[H_tb.atoms.Z==7, 0], xyz[H_tb.atoms.Z==7, 1], 50, c='blue', label="N")

ax.set_xlabel("x [Å]")
ax.set_ylabel("y [Å]")
plt.show()

In [ ]:
es = H_tb.eigenstate(dtype=np.float64).sub([len(H_tb) // 2-1])
grid_spacing = 0.2
g = si.Grid(grid_spacing, dtype=np.float64, geometry=H_tb.geometry)
es.wavefunction(g)

In [ ]:
z_idx = g.shape[2] // 2  # Middle plane along the Z-axis
x, y = np.mgrid[:g.shape[0], :g.shape[1]]
x = x * g.dcell[0, 0] + y * g.dcell[1, 0]
y = y * g.dcell[0, 1] + y * g.dcell[1, 1]

fig, ax = plt.subplots(figsize=(8, 6))
ax.set_aspect('equal')
contour = ax.contourf(x, y, np.abs(g.grid[:, :, z_idx]), cmap="inferno", levels=50)

xyz = H_tb.geometry.xyz
ax.scatter(H_tb.xyz[H_tb.atoms.Z==6, 0], xyz[H_tb.atoms.Z==6, 1], 40, c='gray', label="C")
ax.scatter(H_tb.xyz[H_tb.atoms.Z==7, 0], xyz[H_tb.atoms.Z==7, 1], 50, c='blue', label="N")

ax.set_xlabel("x [Å]")
ax.set_ylabel("y [Å]")
plt.show()

## 4. Transmission along the NPG

For coherent two-terminal transport, the energy- and $k$-resolved transmission is calculated with the Caroli expression

$$T(E,k)=\mathrm{Tr}\left[\boldsymbol\Gamma_L\,\mathbf G^r\,\boldsymbol\Gamma_R\,\mathbf G^a\right],$$

where $\mathbf G^a=(\mathbf G^r)^\dagger$ and

$$\boldsymbol\Gamma_\alpha=i\left(\boldsymbol\Sigma_\alpha-\boldsymbol\Sigma_\alpha^\dagger\right)$$

describes the broadening produced by electrode $\alpha$. The retarded Green function of the open device is

$$\mathbf G^r(E,k)=\left[(E+i\eta)\mathbf S_k-\mathbf H_k-\boldsymbol\Sigma_L(E,k)-\boldsymbol\Sigma_R(E,k)\right]^{-1}.$$

Here $\mathbf H_k$ and $\mathbf S_k$ are the Hamiltonian and overlap matrices at the transverse $k$ point, and $\eta>0$ is a small numerical broadening.

We calculate transport along the **$y$ direction**, corresponding here to lattice vector `B` (axis `1` in `sisl`). The NPG unit cell is repeated three times along this direction to form a finite device containing a left, central, and right cell.

Because the transport direction is now treated as an open direction connected to semi-infinite electrodes, periodicity along $y$ is removed from the finite device by setting the number of supercells in that direction to one.

In [ ]:
#Repeat
H_device=H_tb.tile(3,1)
#nx ny nz
H_device.set_nsc([3, 1, 1])

In [ ]:
H_device.geometry.plot(axes="xy",)

The left and right electrodes are represented with `sisl.RecursiveSI`, which evaluates the surface self-energy of a semi-infinite periodic electrode using the Lopez-Sancho recursive method.

`-B` and `+B` select semi-infinite electrodes extending in the negative and positive directions of lattice vector `B`, respectively.

In [ ]:
SE_bottom = si.RecursiveSI(H_tb,   "-B",    eta=1e-3)
SE_top = si.RecursiveSI( H_tb,"+B",       eta=1e-3)

The NPG is still periodic in the transverse **$x$ direction**. Therefore the transmission depends on the transverse Bloch vector $k_x$, and the physically relevant transmission is obtained by a weighted Brillouin-zone average,

$$
T(E)=\sum_{k_x} w_{k_x}T(E,k_x).
$$

The number of transverse points `nkx` should be increased until the averaged transmission is converged.

### Energy and transverse-$k$ grids

`En` defines the energy grid and `MonkhorstPack(..., [nkx, 1, 1])` samples only the transverse $x$ direction. The transport direction itself is not sampled because it has been opened and replaced by the two semi-infinite electrode self-energies.


In [ ]:
Emin = -1
Emax =  1
dE   =  0.1
eta  =   0.01
nkx = 10

bz = si.MonkhorstPack(H_tb,[nkx, 1, 1])
En   = np.arange(Emin, Emax+dE, dE)

In [ ]:
idx_left=np.arange(0,H_tb.na)
idx_right=np.arange(2*H_tb.na,3*H_tb.na)

### Green-function loop

For every transverse $k_x$, the device Hamiltonian and overlap are evaluated at that same $k$ point. The electrode self-energies must use the same $k_x$. The code then evaluates $T(E,k_x)$ for each energy and stores it before performing the weighted $k_x$ average.


In [ ]:
T_k = np.zeros((len(bz.k), len(En)))
for ik, (k, wk) in enumerate(zip(bz.k, bz.weight)):
    print(f"k-point {ik+1}/{len(bz.k)} "f"k = {k}, weight = {wk}")
    # Device matrices at this transverse kx
    Hk = H_device.Hk(k=k, format="array" )
    Sk = H_device.Sk( k=k, format="array")
    for iE, E in enumerate(tqdm(En, leave=False)):
        Sigma_B = SE_bottom.self_energy(E, k=k)
        Sigma_T = SE_top.self_energy( E, k=k)

        z = E + 1j * eta
        A = z * Sk - Hk

        A[np.ix_(idx_left, idx_left)] -= Sigma_B
        A[np.ix_(idx_right, idx_right)] -= Sigma_T

        G = np.linalg.inv(A)

        Gamma_B = 1j * (  Sigma_B - Sigma_B.conj().T)
        Gamma_T = 1j * ( Sigma_T- Sigma_T.conj().T)

        G_BT = G[np.ix_(idx_left, idx_right)]


        T_k[ik, iE] = np.real( np.trace(Gamma_B@ G_BT @ Gamma_B@ G_BT.conj().T))

In [ ]:
Tavg = np.sum(bz.weight[:, None] * T_k, axis=0)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
ax.plot(En,Tavg,c="k")
ax.set_xlabel("Energy [eV]")
ax.set_ylabel("Transmission")
plt.xlim(-1,1)
plt.ylim(0,None)
plt.show()

# II. Real-space self-energies for graphene

In the previous section, semi-infinite electrodes were attached along a single transport direction. A **real-space self-energy (RSSE)** provides a different type of embedding: it allows a finite region to reproduce the effect of an extended periodic environment around it.

We introduce the method using pristine graphene and a simple nearest-neighbor $p_z$ tight-binding Hamiltonian. The finite region will later be reused as the graphene environment for the graphene–CNT–graphene device.

The RSSE is obtained from the Green function of the periodic reference system. Schematically,

$$\boldsymbol\Sigma^{\mathcal R}(E)=\mathbf S^{\mathcal R}(E+i\eta)-\mathbf H^{\mathcal R}-\left[\sum_{\mathbf k},\mathbf G_{\mathbf k}(E)\right]^{-1}.$$

The Brillouin-zone average carries the information of the infinite periodic graphene sheet into a finite real-space region. In practice, `sisl.RealSpaceSE` handles this construction for us.

## 1. Graphene tight-binding Hamiltonian and band structure

We first construct a graphene unit cell and a nearest-neighbor $p_z$ tight-binding Hamiltonian. Each carbon atom has one orbital, with onsite energy `e0` and nearest-neighbor hopping `t`=-2.7 eV.

The band structure provides a quick check that the model has the expected graphene dispersion before it is used to construct the RSSE.

In [ ]:
graphene = si.geom.graphene(a_cc)
graphene.set_nsc([3,3,1])

In [ ]:
graphene.plot(axes='xy',backend='matplotlib')

In [ ]:
H_tb = si.Hamiltonian(graphene)
r = (0.1, a_cc+0.1)
t_v = (e0, t)
H_tb.construct([r,t_v])
H_tb.set_nsc([3,3,1])

In [ ]:
band_pz = si.BandStructure(H_tb,points= [[0., 0., 0.], [1 / 3, 2 / 3, 0], [0, 0.5, 0],[0., 0., 0.]],divisions= 50, names=[r'$G$', r"$K$", r"$M$",r'$G$'])

In [ ]:
fig = plt.figure('Bands')
ax = fig.add_subplot(111)

lk, kt, kl = band_pz.lineark(True)
ax.set_ylabel(r'$E-E_F$',fontsize=12)

bs = []
for ik, eigh in enumerate(band_pz.apply.eigh()):
    bs.append(eigh)
bs= np.array(bs)

firsttime=True

for bk in bs.T:
    if(firsttime):
        ax.plot(lk, bk, c='r', label=r'$TB$')
        firsttime=False
    else:
        ax.plot(lk, bk, c='r')
ax.hlines(y=0,xmin=0,xmax=10,color='k',lw=0.1)

xcoords = kt
for k in xcoords:
    plt.axvline(x=k,c='k',lw=0.1)
plt.xticks(kt, kl)
ax.legend(loc='center right')
ax.set_xlim(kt[0],kt[-1])
ax.set_xlim(kt[0],kt[-1])

## 2. Construct the real-space self-energy

We now create a `RealSpaceSE` object for an $N	\times N$ graphene region. The periodic tight-binding Hamiltonian provides the reference environment, while the real-space size determines the finite region on which the embedding self-energy is represented.

The $k$-point sampling used in `setup` controls the Brillouin-zone average and should be converged for quantitative calculations.

In [ ]:
N=12
eta = 0.01
Emin = -3
Emax =  3
dE   =  0.1
eta  =   0.01
En   = np.arange(Emin, Emax+dE, dE)

nk1 = int(np.ceil(1000/N))

RSSE = si.RealSpaceSE(H_tb, 0, 1, (N, N, 1), eta = eta)
RSSE.setup(bz=si.MonkhorstPack(H_tb, [1, nk1, 1]))

`real_space_coupling(ret_indices=True)` returns the part of the real-space region that couples to the surrounding periodic environment, together with the corresponding atom indices. These are the atoms on which the coupling form of the RSSE acts.

In [ ]:
H_rs_elec, rs_elec_indices = RSSE.real_space_coupling(ret_indices=True) ## The small one
H_rs_elec.set_nsc([3, 3,1])
#H_rs_elec.write(HOME_DIR+'TBT/Elec.nc')
nC = len(rs_elec_indices)

Next we construct the finite $N	\times N$ graphene device. For convenience, we reorder its atoms so that the RSSE coupling atoms come **first**, followed by the atoms in the interior.

This ordering is not a physical requirement; it simply makes the later matrix operations easier because the RSSE can be inserted directly into the upper-left block of the inverse Green-function matrix.

In [ ]:
# Device Hamiltonian
# equivalent to H = rse.real_space_parent()
HamNN=H_tb.tile(N,0).tile(N,1)
HamNN.set_nsc([1,1,1])
geomNN = HamNN.geometry

# Get indices inside atoms and elec atoms
all_atoms=np.arange(0,HamNN.na)
inside_atoms = np.delete(all_atoms, rs_elec_indices)
Haminside=HamNN.sub(inside_atoms)
HamSE=HamNN.sub(rs_elec_indices)

# reorder indexes
alist = np.concatenate([rs_elec_indices, inside_atoms])
HamNN_reorder=HamNN.sub(alist)
HamNN_reorder.set_nsc([1,1,1])
HamNN_reorder.reduce()
#HamNN_reorder.write(HOME_DIR+'TBT/Device.nc')
na=HamNN_reorder.na

The following plot shows the reordered indexing. The electrode/coupling atoms appear first, while the remaining atoms form the interior of the finite graphene region. Checking this ordering visually is a useful way to avoid index-mapping errors later.

In [ ]:
fig, ax = plt.subplots(figsize=(8,8))
ax.set_aspect('equal')
for i in np.arange(0,HamNN_reorder.na):
    if i < nC:
        ax.text(HamNN_reorder.xyz[i,0],HamNN_reorder.xyz[i,1],i,c='k')
        plt.scatter(HamNN_reorder.xyz[i,0],HamNN_reorder.geometry.xyz[i,1],c='b',s=100,label='Elec')
    else:
        plt.scatter(HamNN_reorder.xyz[i,0],HamNN_reorder.geometry.xyz[i,1],c='gray',s=100,label='Dev')


## 3. DOS and LDOS from the Green function

We now calculate the **density of states (DOS)** and **local density of states (LDOS)** from the retarded Green function of the embedded graphene region.For a non-orthogonal basis, the total DOS is evaluated as:

$$D(E)=-\frac{1}{\pi}\operatorname{Im}\,\mathrm{Tr}\left[\mathbf G^r(E)\mathbf S\right],$$ and the diagonal elements of $\mathbf G^r\mathbf S$ provide an orbital-resolved LDOS. Here each carbon atom has one orbital, so the orbital-resolved LDOS is also atom-resolved.

In [ ]:
Hk= HamNN_reorder.Hk(format="array")
Sk= HamNN_reorder.Sk(format="array")

`coupling=True` returns only the RSSE block acting on the coupling atoms identified above. Because those atoms were reordered to the beginning of the device, this block can be subtracted directly from `A[:n_elec, :n_elec]`.


In [ ]:
def calc_DOS(E,eta,rse,H_sub,S_sub,n_elec):
    z = E +  eta*1j
    A = z*S_sub - H_sub
    # Get RSSE
    RSE = rse.self_energy(E, coupling=True)
    RSE = rse.self_energy(E, coupling=True)
    A[:n_elec, :n_elec] -= RSE
    # Get G
    G = np.linalg.inv(A)
    dos = - (1.0 / np.pi) * np.imag(np.trace(G@S_sub))
    ldos = - (1.0 / np.pi) * np.imag(np.diag(G@S_sub))
    return dos, ldos

def loop_DOS(energies,eta,rse,H_sub,S_sub,n_elec):
    # Observables
    N = H_sub.shape[0]
    DOS = np.zeros_like(energies, dtype=float)
    LDOS = np.zeros((N, len(energies)), dtype=float)

    # Loop
    for iE, E in enumerate(tqdm(energies, desc='DOS')):
        dos,ldos =calc_DOS(E,eta,rse,H_sub,S_sub,n_elec)
        DOS[iE] = dos
        LDOS[:,iE] = ldos
    return DOS, LDOS

In [ ]:
t0 = time.perf_counter()
DOS, LDOS=loop_DOS(En,eta,RSSE,Hk,Sk,nC)
t1 = time.perf_counter()
print(f"Total runtime: {t1 - t0:.2f} seconds")
print(f"Time per energy: {(t1 - t0)/len(En):.4f} s")
LDOS_dev= LDOS[nC:,:]
DOS_dev = LDOS_dev.sum(axis=0)

Finally, we plot the DOS of the interior region. Away from the artificial boundary, the RSSE should make the finite system reproduce the electronic structure of pristine graphene. This provides a useful validation of the embedding before we use the same RSSE in a more complicated device.

In [ ]:
# === plot DOS ===
E_idx=30
plt.figure(figsize=(4,4))
#plt.plot(tbt.E, tbt.DOS(), lw=1.5,c='r',label='TBT')
plt.plot(En,DOS_dev/na,c='k',label='DOS')
plt.xlabel("Energy (eV)")
plt.ylabel("DOS (states / eV)")
plt.vlines(x=En[E_idx],ymin=0,ymax=0.2,color='k',linestyle='dashed')
plt.xlim(-3,3)
plt.ylim(0,None)
plt.legend()
plt.show()

# III. Vertical transport through a carbon nanotube

## 0. Build the graphene–CNT–graphene structure (optional)

This section constructs a CNT connecting two graphene sheets. The structure-building cells are optional:A prepared device geometry is already available, so one can skip directly to the Hamiltonian construction below.

The main geometrical steps are: place the CNT above the graphene, identify the atoms at the CNT ends, optimize the lateral registry with graphene, match CNT end atoms to graphene atoms one-to-one, remove graphene atoms inside the CNT footprint, and finally combine the bottom graphene, CNT, and top graphene into a single geometry.

In [ ]:
def rotate_xy(xy, theta, center):
    c = np.cos(theta)
    s = np.sin(theta)
    R = np.array([[c, -s],[s,  c]])
    return (xy - center) @ R.T + center

def mismatch(params):
    dx, dy, theta = params
    xy = rotate_xy(xy_bottom0,theta, center)
    xy += np.array([dx, dy])
    dist, idx = tree.query(xy)
    return np.sqrt(np.mean(dist**2))

In [ ]:
graphene_bottom=HamNN_reorder.geometry
graphene_top=HamNN_reorder.geometry

We choose a CNT chirality and repeat its unit cell along the tube axis. The following cells align the CNT with the graphene lattice and prepare an atomically connected interface.


In [ ]:
chirality = (6, 6)
cnt = si.geom.nanotube(  bond=a_cc,chirality=chirality)

cnt = cnt.tile(6, 2)
cnt.set_supercell(graphene.cell)

In [ ]:
view(cnt.to.ase())

In [ ]:
# Translate CNT to center
xy_center_cnt = cnt.xyz[:, :2].mean(axis=0)
cnt = cnt.move([ -xy_center_cnt[0],  -xy_center_cnt[1],  0])
xy_center_gr = graphene_bottom.xyz[:, :2].mean(axis=0)
cnt = cnt.move([ xy_center_gr[0],xy_center_gr[1],0])

# Allign CNT to graphene
zmin = cnt.xyz[:, 2].min()
tol_bottom = 0.1
bottom_cnt = np.where( cnt.xyz[:, 2] < zmin +tol_bottom)[0]
print("Number of bottom CNT atoms:", len(bottom_cnt))

z_bottom_gr = graphene_bottom.xyz[:, 2].mean()
z_bottom_cnt = cnt.xyz[bottom_cnt, 2].mean()
cnt = cnt.move([0, 0,   z_bottom_gr - z_bottom_cnt])
xyz = cnt.xyz.copy()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
ax.set_aspect('equal')
plt.scatter(graphene_bottom.xyz[:,0],graphene_bottom.xyz[:,1],c='gray',s=100)
plt.scatter(xyz[bottom_cnt][:,0],xyz[bottom_cnt][:,1],c='r',s=100)

A rigid in-plane translation and rotation are optimized to minimize the RMS mismatch between the projected CNT end atoms and nearby graphene sites. This step improves registry before atoms are matched and merged.


In [ ]:
# Optimize rotation allignement of cnt to graphene
xy_gr = graphene_bottom.xyz[:, :2]
xy_bottom0= cnt.xyz[bottom_cnt, :2].copy()
center = xy_bottom0.mean(axis=0)

a = np.linalg.norm(graphene_bottom.cell[0])
bounds = [ (-a_cc, a_cc), (-a_cc, a_cc),(0, np.pi / 3), ]

tree = cKDTree(xy_gr)
result = differential_evolution( mismatch, bounds=bounds,tol=1e-8,polish=True,seed=1,)
dx_opt, dy_opt, theta_opt = result.x

print("Optimal dx =", dx_opt)
print("Optimal dy =", dy_opt)
print("Optimal theta =", np.degrees(theta_opt), "deg")
print("RMS mismatch =", result.fun, "Ang")

In [ ]:
xyz_rotate = cnt.xyz.copy()
center3d = np.array([ center[0],  center[1],  0.0])
xyz_rotate[:, :2] = rotate_xy( xyz[:, :2], theta_opt,  center)

xyz_rotate[:, 0] += dx_opt
xyz_rotate[:, 1] += dy_opt
cnt.xyz[:] = xyz_rotate

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
ax.set_aspect('equal')
plt.scatter(graphene_bottom.xyz[:,0],graphene_bottom.xyz[:,1],c='gray',s=100)
plt.scatter(xyz_rotate[bottom_cnt][:,0],xyz_rotate[bottom_cnt][:,1],c='b',s=150,alpha=1)

In [ ]:
# Find atoms in graphene within a circle
R_cnt = 0.5 * ( np.max(cnt.xyz[:, 0]) - np.min(cnt.xyz[:, 0]))
xy_center = cnt.xyz[:, :2].mean(axis=0)

r_gr = np.linalg.norm( graphene_bottom.xyz[:, :2] - xy_center, axis=1)
candidate_gr = np.where( r_gr < R_cnt + 1.0 * a_cc)[0]
xy_candidates = graphene_bottom.xyz[candidate_gr, :2]

(`linear_sum_assignment`) enforces a **one-to-one** correspondence between CNT end atoms and candidate graphene atoms, preventing two CNT atoms from being assigned to the same graphene site.


In [ ]:
# Find overlapping atoms in graphene matching with  the atoms in the cnt
D = cdist( cnt.xyz[bottom_cnt, :2],xy_candidates)
row, col = linear_sum_assignment(D)
matched_cnt = bottom_cnt[row]
matched_gr = candidate_gr[col]

matched_dist = D[row, col]

print("One-to-one matching")
print("-------------------")
print("N atoms :", len(matched_cnt))
print("mean    :", matched_dist.mean())
print("RMS     :", np.sqrt(np.mean(matched_dist**2)))
print("min     :", matched_dist.min())
print("max     :", matched_dist.max())

xyz_cnt = cnt.xyz.copy()
# Replace the ovverlapping atoms
xyz_cnt[matched_cnt, 0] = graphene_bottom.xyz[matched_gr, 0]
xyz_cnt[matched_cnt, 1] = graphene_bottom.xyz[matched_gr, 1]
xyz_cnt[matched_cnt, 2] = graphene_bottom.xyz[matched_gr, 2]

cnt.xyz[:] = xyz_cnt

In [ ]:
# Remove atoms in graphene inside the circle of the CNT
cylinder = si.shape.EllipticalCylinder(v=0.9*R_cnt, h = 50., axes=(0,1), center=graphene_bottom.center())
proj_atoms_idx = cylinder.within(graphene_bottom.xyz).nonzero()[0]
print(len(proj_atoms_idx))

concat=np.concatenate((matched_gr, proj_atoms_idx))
remove= np.unique(concat, return_index=True)
remove= np.sort(remove)[0]
print(len(matched_gr),len(proj_atoms_idx),len(remove))
gr_bottom_cut = graphene_bottom.remove(remove)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
ax.set_aspect('equal')
plt.scatter(gr_bottom_cut.xyz[:,0],gr_bottom_cut.xyz[:,1],c='gray',s=100)
plt.scatter(cnt.xyz[:][:,0],cnt.xyz[:][:,1],c='orange',s=100)
plt.scatter(cnt.xyz[bottom_cnt][:,0],cnt.xyz[bottom_cnt][:,1],c='orange',edgecolors='k',s=100)

circle=plt.Circle((gr_bottom_cut.center()[0],gr_bottom_cut.center()[1]), 0.9*R_cnt, color='blue',fill=False)
ax.add_patch(circle)
for idx in np.arange(gr_bottom_cut.na):
    ax.text(gr_bottom_cut.xyz[idx,0],gr_bottom_cut.xyz[idx,1],idx,c='k')
for idx in np.arange(0,len(rs_elec_indices)):
    ax.scatter(gr_bottom_cut.xyz[idx,0],gr_bottom_cut.xyz[idx,1],s=100,c='r')

In [ ]:
# ADD A TOP Graphene
zmax = cnt.xyz[:, 2].max()
tol_top= 0.1
top_cnt = np.where( cnt.xyz[:, 2] >= zmax -tol_top)[0]
print("Number of bottom CNT atoms:", len(top_cnt))

graphene_top=graphene_top.translate([0,0,zmax])

In [ ]:
D = cdist( cnt.xyz[top_cnt, :2],xy_candidates)
row, col = linear_sum_assignment(D)
matched_cnt = top_cnt[row]
matched_gr = candidate_gr[col]
matched_dist = D[row, col]

print("One-to-one matching")
print("-------------------")
print("N atoms :", len(matched_cnt))
print("mean    :", matched_dist.mean())
print("RMS     :", np.sqrt(np.mean(matched_dist**2)))
print("min     :", matched_dist.min())
print("max     :", matched_dist.max())

In [ ]:
xyz_cnt = cnt.xyz.copy()

xyz_cnt[matched_cnt, 0] = graphene_top.xyz[matched_gr, 0]
xyz_cnt[matched_cnt, 1] = graphene_top.xyz[matched_gr, 1]
xyz_cnt[matched_cnt, 2] = graphene_top.xyz[matched_gr, 2]

cnt.xyz[:] = xyz_cnt

In [ ]:
cylinder = si.shape.EllipticalCylinder(v=0.9*R_cnt, h = 50., axes=(0,1), center=graphene_top.center())
proj_atoms_idx = cylinder.within(graphene_top.xyz).nonzero()[0]
print(len(proj_atoms_idx))

concat=np.concatenate((matched_gr, proj_atoms_idx))
remove= np.unique(concat, return_index=True)
remove= np.sort(remove)[0]
print(len(matched_gr),len(proj_atoms_idx),len(remove))
gr_top_cut = graphene_top.remove(remove)

In [ ]:
gr_cnt_gr=gr_bottom_cut.add(cnt.add(gr_top_cut))
view(gr_cnt_gr.to.ase())

In [ ]:
n_bottom = gr_bottom_cut.na
n_cnt    = cnt.na
n_top    = gr_top_cut.na
n_dev    = gr_cnt_gr.na
n_elec = len(rs_elec_indices)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
ax.set_aspect('equal')
plt.scatter(gr_cnt_gr.xyz[:,0],gr_cnt_gr.xyz[:,1],c='gray',s=100)
plt.scatter(cnt.xyz[:][:,0],cnt.xyz[:][:,1],c='orange',s=100)
plt.scatter(cnt.xyz[top_cnt][:,0],cnt.xyz[top_cnt][:,1],c='orange',edgecolors='r',s=100)
plt.scatter(cnt.xyz[bottom_cnt][:,0],cnt.xyz[bottom_cnt][:,1],c='orange',edgecolors='b',s=100)
circle=plt.Circle((gr_bottom_cut.center()[0],gr_bottom_cut.center()[1]), 0.9*R_cnt, color='blue',fill=False)
ax.add_patch(circle)
for idx in np.arange(n_bottom+n_cnt,gr_cnt_gr.na):
    ax.text(gr_cnt_gr.xyz[idx,0],gr_cnt_gr.xyz[idx,1],idx,c='k')
for idx in np.arange(n_bottom+n_cnt,n_bottom+n_cnt+len(rs_elec_indices)):
    ax.scatter(gr_cnt_gr.xyz[idx,0],gr_cnt_gr.xyz[idx,1],s=100,c='r')

In [ ]:
gr_cnt_gr.write("CNT/CNT_device.fdf")

## 1. Construct the device Hamiltonian

We now read the graphene–CNT–graphene structure written to the FDF file and construct its tight-binding Hamiltonian.

The atom ordering is important and was chosen deliberately as

**bottom graphene $\rightarrow$ CNT $\rightarrow$ top graphene**.

Within each graphene block, the RSSE coupling atoms remain first. This lets us reuse the coupling self-energy calculated in Section II without an additional coordinate remapping.

In [ ]:
gr_cnt_gr=si.get_sile("CNT/CNT_device.fdf").read_geometry()

In [ ]:
H_dev = si.Hamiltonian(gr_cnt_gr)

In [ ]:
H_dev.geometry.plot(axes="xyz")

In [ ]:
H_dev = si.Hamiltonian(gr_cnt_gr)
r = (0.1, a_cc+0.1)
t_v = (e0, t)
H_dev.construct([r,t_v])

## 2. Calculate the two-terminal transmission

The bottom and top electrodes are identical pristine graphene environments, so the same RSSE construction can be used for both sides. We keep separate variables (`RSSE_bottom` and `RSSE_top`) to make the two-electrode structure of the calculation explicit.

In [ ]:
RSSE_bottom=RSSE
RSSE_top=RSSE
n_dev    = gr_cnt_gr.na
n_elec = len(rs_elec_indices)

We calculate the transmission from the bottom graphene sheet to the top graphene sheet over a chosen energy range. For each energy we:

1. evaluate the bottom and top coupling self-energies,
2. insert them into the corresponding blocks of the full device matrix,
3. invert the open-system matrix to obtain $\mathbf G^r$,
4. construct the broadening matrices $\boldsymbol\Gamma_B$ and $\boldsymbol\Gamma_T$, and
5.  evaluate
$$T(E)=\mathrm{Tr}\left[\boldsymbol\Gamma_B\mathbf G^r\boldsymbol\Gamma_T\mathbf G^a\right].$$

The same Green function is also used to calculate the total DOS and its projections onto the bottom graphene, CNT, and top graphene regions.

In [ ]:
Emin = -3
Emax =  3
dE   =  0.1
eta  =   0.01
En   = np.arange(Emin, Emax+dE, dE)

In [ ]:
dos_total  = np.zeros(len(En))
dos_bottom = np.zeros(len(En))
dos_cnt    = np.zeros(len(En))
dos_top    = np.zeros(len(En))
ldos       = np.zeros((len(En),n_dev))

In [ ]:
Hk= H_dev.Hk(format="array")
Sk= H_dev.Sk(format="array")

Because the device ordering is **bottom graphene $\rightarrow$ CNT $\rightarrow$ top graphene**.
the global indices of the two RSSE coupling regions follow directly from the sizes of these three blocks.


In [ ]:
# Bottom graphene starts at 0
rs_bottom_global = np.arange(n_elec)

# Top graphene starts after bottom + CNT
top_offset = n_bottom + n_cnt
rs_top_global = top_offset + np.arange(n_elec)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
ax.set_aspect("equal")

ax.scatter(
    gr_cnt_gr.xyz[:, 1],
    gr_cnt_gr.xyz[:, 2],
    c="lightgray",
    s=60
)

ax.scatter(
    gr_cnt_gr.xyz[rs_bottom_global, 1],
    gr_cnt_gr.xyz[rs_bottom_global, 2],
    facecolors="none",
    edgecolors="blue",
    s=130,
    linewidths=2,
    label="Bottom RSSE"
)

ax.scatter(
    gr_cnt_gr.xyz[rs_top_global, 1],
    gr_cnt_gr.xyz[rs_top_global, 2],
    facecolors="none",
    edgecolors="red",
    s=130,
    linewidths=2,
    label="Top RSSE"
)

ax.legend()
ax.set_xlabel("y [Å]")
ax.set_ylabel("z [Å]")
plt.show()

In [ ]:
transmission = np.zeros(len(En))
for iE, E in enumerate(tqdm(En, unit="E")):
    z = E + 1j * eta

    # RSSE
    RSE_b = RSSE_bottom.self_energy(E,coupling=True) # no need to sort
    RSE_t =  RSSE_bottom.self_energy(E,coupling=True)

    A = z * Sk - Hk
    SE_B=np.zeros(A.shape,dtype=complex)
    SE_T=np.zeros(A.shape,dtype=complex)

    SE_B[np.ix_(rs_bottom_global, rs_bottom_global)] -= RSE_b
    SE_T[np.ix_(rs_top_global,rs_top_global) ] -= RSE_t

    A+= (SE_B+SE_T)

    # Greens function
    G = np.linalg.inv(A)

    # Gamma matrices
    Gamma_B = 1j * (SE_B - SE_B.conj().T)
    Gamma_T = 1j * (SE_T - SE_T.conj().T)

    # Transmission
    transmission[iE] = np.real(np.trace( Gamma_B @ G @ Gamma_T@ G.conj().T))
    GS = G @ Sk

    dos_atom = -np.imag( np.diag(GS) ) / np.pi

    dos_total[iE] = dos_atom.sum()

    dos_bottom[iE] = dos_atom[ :n_bottom ].sum()

    dos_cnt[iE] = dos_atom[  n_bottom:n_bottom+n_cnt].sum()

    dos_top[iE] = dos_atom[ n_bottom+n_cnt: ].sum()
    ldos[iE] = - (1.0 / np.pi) * np.imag(np.diag(GS))

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
ax.plot(En,transmission,c='k')
ax.set_xlabel("Energy [eV]")
ax.set_ylabel("Transmission")
plt.xlim(-3,3)
plt.ylim(0,None)
plt.show()

## 3. Visualize the DOS and LDOS

The DOS projections show which part of the device contributes states at a given energy. The LDOS maps provide the complementary real-space picture: they show where those states are localized within the graphene–CNT–graphene structure.

The selected energy index `E_idx` is marked in the DOS plot and then used for the spatial LDOS maps below.

In [ ]:
E_idx=47
print(En[E_idx])

In [ ]:
plt.figure(figsize=(4,4))
plt.plot(En,DOS_dev/na,c='k',label='pristine Graphene')
plt.plot(En,dos_total/n_dev,'-r',label='Device')
plt.plot(En,dos_cnt/n_cnt,'orange',label='CNT')

plt.xlabel("Energy (eV)")
plt.ylabel("DOS (states / eV)")
plt.vlines(x=En[E_idx],ymin=0,ymax=0.2,color='k',linestyle='dashed')
plt.xlim(-3,3)
plt.ylim(0,None)
plt.legend()
plt.show()

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
ax.set_aspect("equal")

sc=ax.scatter(gr_cnt_gr.xyz[:, 0], gr_cnt_gr.xyz[:, 1],  c=ldos[E_idx],  s=1000*ldos[E_idx],cmap='inferno',vmin=0)
ax.scatter(gr_cnt_gr.xyz[rs_bottom_global, 0],gr_cnt_gr.xyz[rs_bottom_global, 1],facecolors="blue", s=130)
ax.scatter( gr_cnt_gr.xyz[rs_top_global, 0], gr_cnt_gr.xyz[rs_top_global, 1],facecolors="red", s=130)

divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="4%",  pad=0.08)
cbar = fig.colorbar( sc,   cax=cax)
cbar.set_label("LDOS [1/eV]")
ax.set_xlabel(r"$x$ [$\AA$]")
ax.set_ylabel(r"$y$ [$\AA$]")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
ax.set_aspect("equal")

sc=ax.scatter(gr_cnt_gr.xyz[:, 0], gr_cnt_gr.xyz[:, 2],  c=ldos[E_idx],  s=1000*ldos[E_idx],cmap='inferno',vmin=0)

divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="4%",  pad=0.08)
cbar = fig.colorbar( sc,   cax=cax)
cbar.set_label("LDOS [1/eV]")


cbar.set_label("LDOS [1/eV]")

ax.set_xlabel(r"$x$ [$\AA$]")
ax.set_ylabel(r"$z$ [$\AA$]")

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
ax.set_aspect("equal")

sc=ax.scatter(gr_cnt_gr.xyz[:, 1], gr_cnt_gr.xyz[:, 2],  c=ldos[E_idx],  s=1000*ldos[E_idx],cmap='inferno',vmin=0)

divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="4%",  pad=0.08)
cbar = fig.colorbar( sc,   cax=cax)
cbar.set_label("LDOS [1/eV]")

ax.set_xlabel(r"$y$ [$\AA$]")
ax.set_ylabel(r"$z$ [$\AA$]")

plt.show()